# 1. Full CAH evaluation

Runs all four models over the whole locked split: 2,607 patches, 12,456
annotated points. Nothing is scored here - this notebook only produces
detections, so it reveals nothing about the outcome. Scoring is `02_metrics`,
which needs no GPU.

Measured on an A100: the probe projects about 5 minutes for OWL-D, the heaviest
model, from a warm cache. The first pass of the full run took 16 minutes,
because each of the 2,607 patches was read from Drive one by one; the other three models, reading from the cache, took under a minute
each. `05_sweep_gpu` stages the split on local disk instead. Standard RAM is
enough; the peak is building OWL-D, around 5 GB.

**fp32 throughout.** bf16 measured 5x faster on the probe (0.116 vs 0.023 s per
patch for OWL-D) but returned 150 detections where fp32 returned 140 on the
same 20 patches (owl-c 162 vs 157, owl-t 108 vs 106). The peak rule thresholds
against each patch's own maximum, so small numeric shifts move points across
the line. With the run this short there is nothing to buy.

Safe to run again: a model whose results exist, produced by this same code, is
skipped. Change the code and the hash changes, so results are recomputed rather
than silently mixed.


In [1]:
# --- 0. Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- 1. Configuration ---
ASSETS  = "/content/drive/MyDrive/OWL_Caribou_Project"
RESULTS = "/content/drive/MyDrive/owl_caribou_overhead/results"

RUN_NAME = "full_cah"
MODELS   = ["owl-d", "caribou-owl-c", "owl-c", "owl-t"]   # primary first
BATCH_SIZE, NUM_WORKERS = 8, 2
FORCE = False     # True recomputes models that already have results

from pathlib import Path
if not (Path(ASSETS) / "data" / "test" / "gt.csv").is_file():
    raise RuntimeError(f"{ASSETS}/data/test/gt.csv not found. Put the assets there as the README's "
                       "'Expected assets' section describes, or point ASSETS at where they are.")


In [ ]:
# === CODE SYNC (auto-generated by `python -m owlcaribou.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m owlcaribou.sync   and reopen this notebook.")

In [4]:
# --- 3. Start the session ---
from owlcaribou.session import start_session

session = start_session(assets=ASSETS, results=RESULTS,
                        models=MODELS, require_gpu=True)

session: Colab | device cuda (NVIDIA A100-SXM4-40GB, 39.49 GB)
  torch   2.11.0+cu128 | CUDA 12.8 | Python 3.13.15
  assets  /content/drive/MyDrive/OWL_Caribou_Project
  results /content/drive/MyDrive/owl_caribou_overhead/results
  code    c26c4d3f466e
  split   2607 patches, 12456 points
  weights verified: caribou-owl-c, owl-c, owl-d, owl-t


In [5]:
# --- 4. The locked split ---
from owlcaribou.data import CahPatches, load_ground_truth, cross_check

patches = CahPatches(session.paths.test_data, expect_images=2607)
truth   = load_ground_truth(session.paths.ground_truth)
print("split:", cross_check(patches, truth))

split: {'patches': 2607, 'with_points': 1852, 'background': 755, 'points': 12456, 'mosaics_with_animals': 26}


In [6]:
# --- 5. Run every model ---
import time
import torch
from owlcaribou import infer, io as oio

summary = []
for model in MODELS:
    out  = session.paths.run_dir(RUN_NAME, model)
    done = oio.is_complete(out, code_hash=session.code_hash,
                           model=model, patches=len(patches))
    if done and not FORCE:
        marker = oio.read_json(out / "_SUCCESS.json")
        print(f"[cached] {model}: {marker['detections']} detections")
        summary.append({"model": model, **{k: marker[k] for k in ("detections", "seconds")}})
        continue
    # An old marker must not vouch for a rerun that stops half-way.
    (out / "_SUCCESS.json").unlink(missing_ok=True)

    print(f"=== {model} ===")
    spec = infer.MODEL_SPECS[model]
    net  = infer.build_model(model, session.third_party, device=session.device)
    info = infer.load_checkpoint(net, session.paths.checkpoint(model),
                                 device=session.device)
    print("  ", info)

    started = time.perf_counter()
    results = list(infer.run_inference(net, patches, spec, device=session.device,
                                       batch_size=BATCH_SIZE, num_workers=NUM_WORKERS))
    elapsed = time.perf_counter() - started

    detections, per_image = infer.to_rows(results)
    oio.write_csv(out / "detections.csv", detections, ["images", "x", "y", "score"])
    oio.write_csv(out / "per_image.csv", per_image, ["images", "count", "peak"])
    session.record(out, model=model, checkpoint=info, prediction_scale=spec.prediction_scale,
                   patches=len(patches), detections=len(detections), seconds=round(elapsed, 1))

    # Written last: its presence means every table above is complete.
    oio.mark_complete(out, code_hash=session.code_hash, model=model,
                      patches=len(patches), detections=len(detections),
                      seconds=round(elapsed, 1))
    print(f"   {len(detections)} detections over {len(patches)} patches "
          f"in {elapsed/60:.1f} min ({elapsed/len(patches):.3f} s/patch)")
    summary.append({"model": model, "detections": len(detections),
                    "seconds": round(elapsed, 1)})

    del net
    torch.cuda.empty_cache()

print()
for row in summary:
    print(f"  {row['model']:16s} {row['detections']:7d} detections  {row['seconds']:7.1f} s")
print()
print("Done. Release the GPU, then run 02_metrics on any runtime.")

=== owl-d ===
   {'checkpoint': 'OWL-D.pth', 'tensors': 672, 'unwrapped': 'model.', 'epoch': 15}
   12494 detections over 2607 patches in 16.0 min (0.369 s/patch)
=== caribou-owl-c ===
   {'checkpoint': 'Caribou-OWL-C.pth', 'tensors': 372, 'unwrapped': 'model.', 'epoch': 14}
   13610 detections over 2607 patches in 0.4 min (0.010 s/patch)
=== owl-c ===
   {'checkpoint': 'OWL-C.pth', 'tensors': 372, 'unwrapped': 'model.', 'epoch': 18}
   12774 detections over 2607 patches in 0.4 min (0.010 s/patch)
=== owl-t ===


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


   {'checkpoint': 'OWL-T.pth', 'tensors': 512, 'unwrapped': 'model.', 'epoch': 10}
   11976 detections over 2607 patches in 0.4 min (0.010 s/patch)

  owl-d              12494 detections    962.3 s
  caribou-owl-c      13610 detections     25.6 s
  owl-c              12774 detections     25.8 s
  owl-t              11976 detections     26.5 s

Done. Release the GPU, then run 02_metrics on any runtime.


## What was written

Per model, under `results/full_cah/<model>/`:

| File | Contents |
|---|---|
| `detections.csv` | one row per detection: patch, x, y, score, in patch pixels |
| `per_image.csv` | every patch including the 755 empty ones, with its count and peak |
| `provenance.json` | code hash, GPU, torch version, checkpoint digest, timings |
| `_SUCCESS.json` | written last; its absence means the model is incomplete |

Detection coordinates are in **patch pixels** (0-511), matching the ground
truth. Tables written by the upstream evaluator store heatmap indices
instead, needing a factor of the `down_ratio`, a difference worth
remembering if the two are ever compared directly.

**Next:** `02_metrics.ipynb`. It needs no GPU, so disconnect this runtime first.
